# ITS AI Codellama-7B-Instruct QLoRA finetuning

NOTE: Before we begin, ensure you already have access to T4 GPU in Kaggle. You can do so by verification using phone number or persona verification. If not, you won't be able to train the codellama using a GPU.

[1] First we need ensure the current environment session is running on T4 GPU and we need to install the packages first:

In [1]:
!nvidia-smi

!pip install -q --no-cache-dir \
  transformers==4.46.3 \
  accelerate==1.1.1 \
  peft==0.14.0 \
  trl==0.12.2 \
  bitsandbytes==0.48.1 \
  datasets==3.1.0 \
  huggingface_hub==0.26.2

!pip uninstall -y triton

Wed Jul 29 08:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[2] Next, import the datasets on the right sidebar. Currently we have to import soal_ujian.json and nilai_ujian.json, the datasets that has been processed by Harmoni (shoutout to harmoni).
If you already have processed data (e.g. train_data.jsonl), upload that as well

[3] Login to HuggingFace via Kaggle Secrets
Add-ons -> Secrets -> add a secret named HF_TOKEN with your own Hugging Face token. This is required because codellama-7b-instruct model is gated by HF

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

[4] Next we are going to check the datasets folder pathname first

In [ ]:
# CHECK THE ACTUAL FOLDER/FILE PATHNAME FIRST

import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

[5] then we're going to rebuild the datasets. the purpose of this is to adjust with the finetuning data format

In [ ]:
import json

# change the path according to your preferred set filepath
with open("/kaggle/input/datasets/hanzfr/soal-ujian/soal_ujian.json") as f:
    soal = json.load(f)
with open("/kaggle/input/datasets/hanzfr/nilai-ujian/nilai_ujian.json") as f:
    nilai = json.load(f)

soal_by_id = {s["id"]: s for s in soal}

def normalize_scores(nilai_dict):
    if max(nilai_dict.values()) <= 10:
        return {k: v * 10 for k, v in nilai_dict.items()}
    return nilai_dict

dataset = []
for n in nilai:
    q = soal_by_id.get(n["id_soal"])
    if q is None:
        continue
    scores = normalize_scores(n["nilai"])
    avg = round(sum(scores.values()) / len(scores), 2)
    dataset.append({
        "id_soal": n["id_soal"],
        "soal": q["soal"],
        "expected_output": q["expected_output"],
        "kode_siswa": n["kode_siswa"],
        "level_siswa": n["level_siswa"],
        "nilai": scores,
        "nilai_avg": avg,
        "feedback": n["feedback"],
    })

print(f"Usable examples: {len(dataset)}")

def format_example(ex):
    prompt = (
        f"Soal: {ex['soal']}\n"
        f"Output yang diharapkan: {ex['expected_output']}\n\n"
        f"Kode siswa:\n```python\n{ex['kode_siswa']}\n```\n\n"
        f"Nilai kode siswa ini dan berikan feedback."
    )
    response = (
        f"Penilaian:\n"
        + "\n".join(f"- {k}: {v}" for k, v in ex["nilai"].items())
        + f"\n\nRata-rata: {ex['nilai_avg']}\n\nFeedback: {ex['feedback']}"
    )
    return {"id_soal": ex["id_soal"], "text": f"<s>[INST] {prompt} [/INST] {response} </s>"}

formatted = [format_example(ex) for ex in dataset]

with open("/kaggle/working/train_data.jsonl", "w") as f:
    for row in formatted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

[6] Check token lengths from the datasets. we will ensure there's no dataset that is exceeding the context window

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-Instruct-hf")

lengths = []
with open("/kaggle/working/train_data.jsonl") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tokenizer(row["text"])["input_ids"]))

import statistics
print("min/max:", min(lengths), max(lengths))
print("mean:", statistics.mean(lengths))
print("p95:", sorted(lengths)[int(len(lengths)*0.95)])

[7] split the dataset into training and validation. we have to make sure codellama TRULY learns to generalize and not memorizing the training examples. preventing overfitting

In [ ]:
import random

rows = [json.loads(l) for l in open("/kaggle/working/train_data.jsonl")]
ids = sorted(set(r["id_soal"] for r in rows))
random.seed(42)
random.shuffle(ids)
val_ids = set(ids[:int(len(ids)*0.15)])

train_rows = [r for r in rows if r["id_soal"] not in val_ids]
val_rows = [r for r in rows if r["id_soal"] in val_ids]
print(f"train: {len(train_rows)}, val: {len(val_rows)}")

with open("/kaggle/working/train_split.jsonl", "w") as f:
    for r in train_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
with open("/kaggle/working/val_split.jsonl", "w") as f:
    for r in val_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback


MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

dataset = load_dataset("json", data_files={
    "train": "/kaggle/working/train_split.jsonl",
    "validation": "/kaggle/working/val_split.jsonl",
})

training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=6,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=6,
    load_best_model_at_end=True,
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=1024,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.save_model("/kaggle/working/final_model")

# POST FINETUNING
run these cells below after finetuning the codellama

In [8]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/adapter_model.safetensors
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/train_data.jsonl
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/training_args.bin
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/adapter_config.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/README.md
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/tokenizer.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/tokenizer_config.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/__huggingface_repos__.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/train_split.jsonl
/kaggle/input/datasets/hanzfr/codellama-q

In [ ]:
!pip install -q --no-cache-dir --upgrade peft

In [ ]:
!pip show bitsandbytes | grep Version

In [2]:
import bitsandbytes as bnb
print(bnb.__version__)

0.48.1


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

### Cell below is for evaluating on only 1 example from val_split.jsonl

In [11]:
val_rows = [json.loads(l) for l in open("/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl")]
test_row = val_rows[0]
print(test_row)

{'id_soal': 5, 'text': "<s>[INST] Soal: Buat fungsi untuk menghitung keliling persegi.\nOutput yang diharapkan: Jika sisi=4, maka keliling=16\n\nKode siswa:\n```python\nsisi = 4\nhasil = 4 + 4 + 4 + 4\nprint(hasil)\n```\n\nNilai kode siswa ini dan berikan feedback. [/INST] Penilaian:\n- fungsionalitas: 20\n- logika: 25\n- syntax: 70\n- code_style: 15\n- dokumentasi: 0\n- konsep: 15\n\nRata-rata: 24.17\n\nFeedback: Kode ini belum memenuhi permintaan soal karena tidak dibuat dalam bentuk fungsi, padahal soal secara eksplisit meminta 'buat fungsi untuk menghitung keliling persegi'. Nilai hasil juga ditulis manual (4+4+4+4) bukan dihitung dari variabel 'sisi', sehingga kode tidak bisa dipakai untuk sisi lain selain 4 — ini menunjukkan siswa belum memahami konsep parameterisasi. Tidak ada dokumentasi/komentar sama sekali. Sarannya: pelajari cara membuat fungsi dengan def nama_fungsi(parameter):, lalu gunakan parameter tersebut dalam perhitungan, misalnya sisi * 4, agar fungsi bisa digunakan

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Instruct-hf",
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

MODEL_PATH = "/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model"

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# grab one held-out example from val_split.jsonl to sanity-check
import json
val_rows = [json.loads(l) for l in open("/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl")]
test_row = val_rows[0]

# rebuild just the prompt half (before [/INST])
prompt = test_row["text"].split("[/INST]")[0] + "[/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[INST] Soal: Buat fungsi untuk menghitung keliling persegi.
Output yang diharapkan: Jika sisi=4, maka keliling=16

Kode siswa:
```python
sisi = 4
hasil = 4 + 4 + 4 + 4
print(hasil)
```

Nilai kode siswa ini dan berikan feedback. [/INST] Penilaian:
- fungsionalitas: 10
- logika: 15
- syntax: 85
- code_style: 25
- dokumentasi: 0
- konsep: 10

Rata-rata: 20.83

Feedback: Siswa mulai memahami konsep inti dari keliling persegi (4 + 4 + 4 + 4 = 16), namun belum menyusun fungsi sama sekali. Ini bukan kondisi yang menunjukkan pemahaman dasar di bidang pemrograman. Tidak ada variabel sisa (sisi) dan tidak ada pesan yang jelas mengingatkan bahwa sisa adalah 4, sehingga hasilnya akan berbeda untuk sisi berbeda. Tidak ada dokumentasi kode sama sekali. Sarannya: belajar membuat fungsi dengan parameter dan return value yang jelas, seperti 'def persegi(sisi): return sisi*4'. 

Saran: Belajar konsep dasar pemrograman seperti variabel, operator, dan penamaan


### Cell below is for evaluating over 10 first validation examples data (run the cell above first)

In [14]:
import json, re

val_rows = [json.loads(l) for l in open("/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl")]

def extract_scores(text):
    """Pull out the 6 rubric scores as a dict from generated or ground-truth text."""
    fields = ["fungsionalitas", "logika", "syntax", "code_style", "dokumentasi", "konsep"]
    scores = {}
    for f in fields:
        m = re.search(rf"{f}:\s*(\d+)", text)
        if m:
            scores[f] = int(m.group(1))
    return scores

results = []

for i, row in enumerate(val_rows[:10]):
    full_text = row["text"]
    prompt = full_text.split("[/INST]")[0] + "[/INST]"
    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_full = tokenizer.decode(output[0], skip_special_tokens=True)
    generated = generated_full.split("[/INST]")[1].strip() if "[/INST]" in generated_full else generated_full

    gt_scores = extract_scores(ground_truth)
    gen_scores = extract_scores(generated)

    gt_avg = sum(gt_scores.values()) / len(gt_scores) if gt_scores else None
    gen_avg = sum(gen_scores.values()) / len(gen_scores) if gen_scores else None

    results.append({
        "id_soal": row["id_soal"],
        "gt_avg": gt_avg,
        "gen_avg": gen_avg,
        "diff": (gen_avg - gt_avg) if (gt_avg and gen_avg) else None,
        "gt_scores": gt_scores,
        "gen_scores": gen_scores,
    })

    print(f"--- Example {i+1} (id_soal={row['id_soal']}) ---")
    print(f"Ground truth avg: {gt_avg} | Model avg: {gen_avg} | Diff: {results[-1]['diff']}")
    print()

# Summary
import statistics
diffs = [r["diff"] for r in results if r["diff"] is not None]
print("=" * 50)
print(f"Mean diff (model - ground truth): {statistics.mean(diffs):.2f}")
print(f"Stdev of diff: {statistics.stdev(diffs):.2f}" if len(diffs) > 1 else "")
print(f"Model scores higher than GT in {sum(1 for d in diffs if d > 0)}/{len(diffs)} cases")
print(f"Model scores lower than GT in {sum(1 for d in diffs if d < 0)}/{len(diffs)} cases")

--- Example 1 (id_soal=5) ---
Ground truth avg: 24.166666666666668 | Model avg: 21.666666666666668 | Diff: -2.5

--- Example 2 (id_soal=5) ---
Ground truth avg: 24.166666666666668 | Model avg: 30.833333333333332 | Diff: 6.666666666666664

--- Example 3 (id_soal=5) ---
Ground truth avg: 28.333333333333332 | Model avg: 23.333333333333332 | Diff: -5.0

--- Example 4 (id_soal=5) ---
Ground truth avg: 20.0 | Model avg: 23.333333333333332 | Diff: 3.333333333333332

--- Example 5 (id_soal=5) ---
Ground truth avg: 33.333333333333336 | Model avg: 33.333333333333336 | Diff: 0.0

--- Example 6 (id_soal=5) ---
Ground truth avg: 24.166666666666668 | Model avg: 26.666666666666668 | Diff: 2.5

--- Example 7 (id_soal=5) ---
Ground truth avg: 49.166666666666664 | Model avg: 32.5 | Diff: -16.666666666666664

--- Example 8 (id_soal=5) ---
Ground truth avg: 52.5 | Model avg: 54.666666666666664 | Diff: 2.1666666666666643

--- Example 9 (id_soal=5) ---
Ground truth avg: 47.5 | Model avg: 33.333333333333336 

### Cell below is for evaluating over 10 broad/random validation examples data (run the cell above first)

In [16]:
import json, re, random, statistics

val_rows = [json.loads(l) for l in open("/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl")]

def extract_scores(text):
    """Pull out the 6 rubric scores as a dict from generated or ground-truth text."""
    fields = ["fungsionalitas", "logika", "syntax", "code_style", "dokumentasi", "konsep"]
    scores = {}
    for f in fields:
        m = re.search(rf"{f}:\s*(\d+)", text)
        if m:
            scores[f] = int(m.group(1))
    return scores

# --- sample one example per distinct id_soal, across as many questions as available ---
random.seed(1)
unique_ids = list(set(r["id_soal"] for r in val_rows))
sampled_ids = random.sample(unique_ids, min(10, len(unique_ids)))

sample_rows = []
for qid in sampled_ids:
    matching = [r for r in val_rows if r["id_soal"] == qid]
    sample_rows.append(random.choice(matching))

print(f"Sampled {len(sample_rows)} examples across {len(sampled_ids)} distinct question ids: {sampled_ids}\n")

results = []
for i, row in enumerate(sample_rows):
    full_text = row["text"]
    prompt = full_text.split("[/INST]")[0] + "[/INST]"
    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_full = tokenizer.decode(output[0], skip_special_tokens=True)
    generated = generated_full.split("[/INST]")[1].strip() if "[/INST]" in generated_full else generated_full

    gt_scores = extract_scores(ground_truth)
    gen_scores = extract_scores(generated)

    gt_avg = sum(gt_scores.values()) / len(gt_scores) if gt_scores else None
    gen_avg = sum(gen_scores.values()) / len(gen_scores) if gen_scores else None

    results.append({
        "id_soal": row["id_soal"],
        "gt_avg": gt_avg,
        "gen_avg": gen_avg,
        "diff": (gen_avg - gt_avg) if (gt_avg and gen_avg) else None,
        "gt_scores": gt_scores,
        "gen_scores": gen_scores,
    })

    print(f"--- Example {i+1} (id_soal={row['id_soal']}) ---")
    print(f"Ground truth avg: {gt_avg} | Model avg: {gen_avg} | Diff: {results[-1]['diff']}")
    print()

# Summary
diffs = [r["diff"] for r in results if r["diff"] is not None]
print("=" * 50)
print(f"Mean diff (model - ground truth): {statistics.mean(diffs):.2f}")
print(f"Stdev of diff: {statistics.stdev(diffs):.2f}" if len(diffs) > 1 else "")
print(f"Model scores higher than GT in {sum(1 for d in diffs if d > 0)}/{len(diffs)} cases")
print(f"Model scores lower than GT in {sum(1 for d in diffs if d < 0)}/{len(diffs)} cases")

Sampled 6 examples across 6 distinct question ids: [34, 12, 33, 46, 11, 5]

--- Example 1 (id_soal=34) ---
Ground truth avg: 90.66666666666667 | Model avg: 85.16666666666667 | Diff: -5.5

--- Example 2 (id_soal=12) ---
Ground truth avg: 95.5 | Model avg: 93.0 | Diff: -2.5

--- Example 3 (id_soal=33) ---
Ground truth avg: 89.66666666666667 | Model avg: 96.66666666666667 | Diff: 7.0

--- Example 4 (id_soal=46) ---
Ground truth avg: 75.66666666666667 | Model avg: 87.5 | Diff: 11.833333333333329

--- Example 5 (id_soal=11) ---
Ground truth avg: 63.666666666666664 | Model avg: 74.5 | Diff: 10.833333333333336

--- Example 6 (id_soal=5) ---
Ground truth avg: 97.16666666666667 | Model avg: 92.83333333333333 | Diff: -4.333333333333343

Mean diff (model - ground truth): 2.89
Stdev of diff: 7.89
Model scores higher than GT in 3/6 cases
Model scores lower than GT in 3/6 cases


# MERGE THE MODEL TO PREPARE TESTING ON LOCAL

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Load base model WITHOUT 4-bit quantization this time — merging requires full precision
base_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Instruct-hf",
    torch_dtype=torch.float16,
    device_map={"": 0},
)

MODEL_PATH = "/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model"
model = PeftModel.from_pretrained(base_model, MODEL_PATH)

merged_model = model.merge_and_unload()
merged_model.save_pretrained("/kaggle/working/merged_model", safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained("/kaggle/working/merged_model")

config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]